In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import torch
from gluonts.dataset.util import to_pandas

from tsfm_lens.dataset import GiftEvalDataset
from tsfm_lens.metrics import compute_metrics
from tsfm_lens.toto.circuitlens import CircuitLensToto
from tsfm_lens.utils import (
    apply_custom_style,
    load_dyst_samples,
    set_seed,
)

In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

In [ ]:
device = "cuda:1" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../../figs", "ablations_toto", "examples")
os.makedirs(plot_save_dir, exist_ok=True)

### Load Pipeline

In [ ]:
# Load the pre-trained model
pipeline = CircuitLensToto("Datadog/Toto-Open-Base-1.0", device_map=device)

In [ ]:
num_layers = pipeline.num_layers
num_heads = pipeline.num_heads

print(f"num_layers: {num_layers}, num_heads: {num_heads}")

In [ ]:
# pipeline.model

In [ ]:
model_param_memory_mb = sum(p.numel() * p.element_size() for p in pipeline.model.parameters()) / (1024 * 1024)
print(f"Model parameter memory: {model_param_memory_mb:.2f} MB")

### Load Data

In [ ]:
split_name = "gift-eval"
subsample_interval = 1

if split_name == "base40":
    split_dir = os.path.join(DATA_DIR, split_name)
    system_name = "Thomas"

    context_length = 512
    context_start_time = 2000
    context_end_time = context_start_time + context_length

    sample_idx = 0
    selected_dim = [0, 1, 2]

    dyst_coords = load_dyst_samples(system_name, split_dir, one_dim_target=False, num_samples=1)
    dyst_coords = torch.tensor(dyst_coords[sample_idx, selected_dim, :])
    print(dyst_coords.shape)
    dyst_coords = dyst_coords[:, ::subsample_interval]
    context = dyst_coords[:, context_start_time:context_end_time]
    print(context.shape)

    prediction_length = 512
    future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]
    print(f"future_vals shape: {future_vals.shape}")

elif split_name == "gift-eval":
    split_dir = os.path.join(DATA_DIR, split_name)
    # system_name = "ett1/15T"
    # system_name = "bizitobs_service"
    system_name = "LOOP_SEATTLE/H"
    to_univariate = False
    term = "medium"

    dataset = GiftEvalDataset(name=system_name, term=term, to_univariate=to_univariate, data_dir=split_dir)
    print("Prediction length: ", dataset.test_data.prediction_length)
    print(f"Number of windows: {dataset.windows}")
    print(f"Number of rows in the dataset: {dataset.hf_dataset.num_rows}")

    # test_data_iter = iter(dataset.test_data)
    # test_data = next(test_data_iter)
    context_iter = iter(dataset.test_data.input)
    future_iter = iter(dataset.test_data.label)
    selected_sample_idx = 314
    for _ in range(selected_sample_idx + 1):
        context_entry = next(context_iter)
        future_entry = next(future_iter)

    print("Keys in the context entry: ", context_entry.keys())
    print("\n\nContext Item id: ", context_entry["item_id"])
    print("Context Start Date: ", context_entry["start"])
    print("Context Frequency: ", context_entry["freq"])
    print(f"Context shape: {context_entry['target'].shape}")
    print("\n\nForecast Item id: ", future_entry["item_id"])
    print("Forecast Start Date: ", future_entry["start"])
    print("Forecast Frequency: ", future_entry["freq"])
    print(f"Forecast shape: {future_entry['target'].shape}")
    context_target = context_entry["target"]
    future_target = future_entry["target"]

    print(f"context target shape: {context_target.shape}")
    print(f"future target shape: {future_target.shape}")
    ndims = context_target.ndim
    if ndims > 1:
        # selected_dim = [0, 1, 2]
        selected_dim = [0, 1]
        print(f"selected dim: {selected_dim}")
        context_target = context_target[selected_dim]
        future_target = future_target[selected_dim]
    else:
        context_target = context_target[None, ...]
        future_target = future_target[None, ...]
    print(f"selected dims context target shape: {context_target.shape}")
    print(f"selected dims future target shape: {future_target.shape}")
    print(context_entry["target"].shape)

    num_variates = context_target.shape[0]
    fig, axes = plt.subplots(num_variates, 1, figsize=(5, 2 * num_variates))
    if num_variates == 1:
        axes = [axes]

    for dim, ax in enumerate(axes):
        context_series = to_pandas({"target": context_target[dim], "start": context_entry["start"]})
        future_series = to_pandas({"target": future_target[dim], "start": future_entry["start"]})

        context_series.plot(color="black", linewidth=1, ax=ax, label="Context")
        future_series.plot(color="tab:blue", linewidth=1, ax=ax, label="Ground Truth")
        ax.grid(which="both")
        ax.set_title(f"Dimension {dim}")
        ax.legend()

    plt.tight_layout()
    plt.show()

    num_nans_context = sum(
        to_pandas({"target": context_target[dim], "start": context_entry["start"]}).isna().sum()
        for dim in range(num_variates)
    )
    num_nans_future_vals = sum(
        to_pandas({"target": future_target[dim], "start": future_entry["start"]}).isna().sum()
        for dim in range(num_variates)
    )
    prediction_length = future_target.shape[1]

    # convert to torch tensors
    context = torch.tensor(context_target)
    future_vals = torch.tensor(future_target)
    print(f"num nans context: {num_nans_context}")
    print(f"num nans future vals: {num_nans_future_vals}")
    print(f"context shape: {context.shape}")
    print(f"future vals shape: {future_vals.shape}")

Extra processing for Toto MaskedTimeseries requirement

In [ ]:
# # # Prepare your input time series (channels, time_steps)
# # context = torch.randn(3, 4096).to(device)  # Example with 7 variables and 4096 timesteps

# context = context.to(device)
# print(f"context shape: {context.shape}")

# ### Dummy variables for future forward compatibility ###
# # Prepare timestamp information (optional, but expected by API; not used by the current model release)
# timestamp_seconds = torch.zeros_like(context).to(device)
# # time_interval_seconds = torch.full_like((3,), 60 * 15)  # 15-minute intervals
# time_interval_seconds = torch.full((context.shape[0],), 60 * 15).to(device)  # 15-minute intervals

# print(f"timestamp_seconds shape: {timestamp_seconds.shape}")
# print(f"time_interval_seconds shape: {time_interval_seconds.shape}")

# # Create a MaskedTimeseries object
# inputs = MaskedTimeseries(
#     series=context,
#     padding_mask=torch.full_like(context, True, dtype=torch.bool),
#     id_mask=torch.zeros_like(context),
#     timestamp_seconds=timestamp_seconds,
#     time_interval_seconds=time_interval_seconds,
# )

In [ ]:
# inputs.time_interval_seconds.shape

### Ablations

In [ ]:
import json

ASSETS_DIR = os.path.join("../../", "assets")

In [ ]:
pipeline.remove_all_hooks()

layers_to_ablate = [2, 8, 9, 10]
ablate_n_heads_per_layer = 6
ablate_by_high_to_low_srank = True
ablations_types = ["head"]
model_srank_per_layer_path = os.path.join(ASSETS_DIR, f"{pipeline.__class__.__name__}_srank_low_to_high_by_layer.json")
# if not exists, raise error
if not os.path.exists(model_srank_per_layer_path):
    raise ValueError(f"Srank per layer file not found at {model_srank_per_layer_path}")
srank_per_layer = json.load(open(model_srank_per_layer_path))
heads_to_ablate = []
for layer in layers_to_ablate:
    if ablate_by_high_to_low_srank:
        heads_to_ablate.extend(
            [
                (int(layer), int(head))
                for head in list(srank_per_layer[str(layer)].keys())[::-1][:ablate_n_heads_per_layer]
            ]
        )
    else:
        heads_to_ablate.extend(
            [(int(layer), int(head)) for head in list(srank_per_layer[str(layer)].keys())[:ablate_n_heads_per_layer]]
        )

print(f"Ablating heads: {heads_to_ablate}")

pipeline.add_ablation_hooks_explicit(
    ablations_types=ablations_types,  # type: ignore
    layers_to_ablate_mlp=layers_to_ablate,
    heads_to_ablate=heads_to_ablate,
)


In [ ]:
layers_without_heads = list(pipeline.head_ablation_handles.keys())
layers_without_mlps = list(pipeline.mlp_ablation_handles.keys())

if layers_without_heads and layers_without_mlps:
    if layers_without_heads != layers_without_mlps:
        raise NotImplementedError("Zero-ablation of heads and MLPs in different layers is messier, save for later")
    ablations_summary_str_title = f"Zero-Ablation: Heads and MLPs in Layers {layers_without_heads}"
    ablations_summary_str = f"za_heads_mlps_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_heads:
    ablations_summary_str_title = f"Zero-Ablation: Heads in Layers {layers_without_heads}"
    ablations_summary_str = f"za_heads_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_mlps:
    ablations_summary_str_title = f"Zero-Ablation: MLPs in Layers {layers_without_mlps}"
    ablations_summary_str = f"za_mlps_layers_{'-'.join(map(str, layers_without_mlps))}"
else:
    ablations_summary_str = ablations_summary_str_title = None

print(ablations_summary_str)
print(ablations_summary_str_title)


In [ ]:
dict(pipeline.model.transformer.layers[7].attention.wO._forward_hooks)

In [ ]:
pipeline.head_ablation_handles

### Predict and Get Outputs

In [ ]:
context.shape

In [ ]:
rseed = 123
num_samples = 20

set_seed(rseed)

pred = pipeline.predict(
    context,
    prediction_length=prediction_length,
    num_samples=num_samples,  # Number of samples for probabilistic forecasting
    samples_per_batch=num_samples,  # Control memory usage during inference
)

In [ ]:
pred.shape

In [ ]:
pipeline.model.patch_embed.projection.in_features

### Plot Predictions with Ablations

In [ ]:
plot_samples = False

In [ ]:
# Prepare context and predictions
context_np = context.cpu().numpy()
context_timesteps = np.arange(context_np.shape[-1])
future_vals_np = future_vals
future_timesteps = np.arange(context_np.shape[-1], context_np.shape[-1] + prediction_length)
print(f"context_np shape: {context_np.shape}")
print(f"future_vals_np shape: {future_vals_np.shape}")
print(f"length of context_timesteps: {len(context_timesteps)}")
print(f"length of future_timesteps: {len(future_timesteps)}")

In [ ]:
preds = pred.squeeze().detach().cpu().numpy()
print(f"preds shape: {preds.shape}")
if preds.ndim == 2:
    preds = preds[None, ...]

num_variates, num_samples, prediction_length = preds.shape

fig, axs = plt.subplots(num_variates, 1, figsize=(6, 2 * num_variates))
axs = [axs] if num_variates == 1 else axs

for dim, ax in enumerate(axs):
    ax.plot(context_timesteps[-512:], context_np[dim][-512:], "k-", linewidth=1, label="Context")
    ax.plot(future_timesteps, future_vals_np[dim], "k--", linewidth=1, label="Ground Truth", zorder=10)
    if plot_samples:
        for i in range(num_samples):
            ax.plot(future_timesteps, preds[dim, i], linewidth=1, color=DEFAULT_COLORS[i], alpha=0.2)
    ax.plot(future_timesteps, np.median(preds, axis=1)[dim], color="tab:blue", linewidth=1, label="Median", zorder=11)
    ax.set_xlabel("Timestep", fontweight="bold")
    ax.set_ylabel(f"Dimension {dim}", fontweight="bold")
    ax.legend(loc="lower left", frameon=True)

plt.tight_layout()

### Plot Original Predictions

In [ ]:
pipeline.remove_all_hooks()
set_seed(rseed)

pred_original = pipeline.predict(
    context,
    prediction_length=prediction_length,
    num_samples=num_samples,  # Number of samples for probabilistic forecasting
    samples_per_batch=num_samples,  # Control memory usage during inference
)

In [ ]:
preds_original = pred_original.squeeze().detach().cpu().numpy()
print(f"preds_original shape: {preds_original.shape}")
if preds_original.ndim == 2:
    preds_original = preds_original[None, ...]

num_variates, num_samples, prediction_length = preds_original.shape

fig, axs = plt.subplots(num_variates, 1, figsize=(6, 2 * num_variates))
axs = [axs] if num_variates == 1 else axs

for dim, ax in enumerate(axs):
    ax.plot(context_timesteps[-512:], context_np[dim][-512:], "k-", linewidth=1, label="Context")
    ax.plot(future_timesteps, future_vals_np[dim], "k--", label="Ground Truth", linewidth=1, zorder=10)
    if plot_samples:
        for i in range(num_samples):
            ax.plot(future_timesteps, preds_original[dim, i], linewidth=1, color=DEFAULT_COLORS[i], alpha=0.2)
    ax.plot(
        future_timesteps,
        np.median(preds_original, axis=1)[dim],
        color="tab:blue",
        linewidth=1,
        label="Median",
        zorder=11,
    )
    ax.set_xlabel("Timestep", fontweight="bold")
    ax.set_ylabel(f"Dimension {dim}", fontweight="bold")
    ax.legend(loc="lower left", frameon=True)

plt.tight_layout()


### Compute Metrics

In [ ]:
med_pred = np.median(preds, axis=0)
med_orig = np.median(preds_original, axis=0)
print(f"med_pred shape: {med_pred.shape}")
print(f"med_orig shape: {med_orig.shape}")

In [ ]:
pred_horizon_lst = [64]
for pred_horizon in pred_horizon_lst:
    print(f"Computing metrics for prediction horizon {pred_horizon}")
    metrics_combined = compute_metrics(med_orig[:pred_horizon], med_pred[:pred_horizon])
    pprint(metrics_combined, width=1, indent=2)